# Cleaning IPCEI firm data

In [1]:
import os
import pandas as pd

# 📂 Set working directory
WORKING_DIR = "/Users/go82gax/Documents/Projekte/IPCEI/Analyse/Daten/New"
os.chdir(WORKING_DIR)

# 📥 Load the IPCEI dataset
ipcei_df = pd.read_csv("IPCEI_combined.csv")

1. Re-code countries to match LinkedIn data

In [18]:
# 📌 Mapping of country names to 3-letter ISO country codes
country_to_code = {
    "Germany": "DE",
    "France": "FR",
    "Italy": "IT",
    "Belgium": "BE",
    "Austria": "AT",
    "Netherlands": "NL",
    "Portugal": "PT",
    "Spain": "ES",
    "Denmark": "DK",
    "Sweden": "SE",
    "Finland": "FI",
    "Luxembourg": "LUX",
    "Greece": "GR",
    "Poland": "PO",
    "United Kingdom": "GB",
    "Japan": "JP",
    "USA": "US",
    "Slovakia": "SK",
    "Estonia": "EE",
    "Switzerland": "CH",
    "Norway": "NO",
    "Ireland": "IE",
    "Czech Republic": "CZ",
    "Hungary": "HU",
    "Lithuania": "LT",
    "Latvia": "LV",
    "Romania": "RO",
    "Bulgaria": "BG",
    "Slovenia": "SV",
    "Croatia": "HV"
}

# 🔹 Function to replace country names with their 3-letter codes
def convert_countries(country_entry):
    if pd.isna(country_entry):
        return ""  # Handle missing values

    country_list = country_entry.split(", ")  # Split multiple countries
    converted_list = [country_to_code.get(country.strip(), country.strip()) for country in country_list]

    return ", ".join(converted_list)  # Join back into a single string

# ✅ Apply the conversion function to the "country" column
ipcei_data["country"] = ipcei_data["country"].apply(convert_countries)

# 📤 Save the updated dataset
updated_file_path = os.path.join(WORKING_DIR, "IPCEI_combined_country_codes.csv")
ipcei_data.to_csv(updated_file_path, index=False)

print(f"✅ Updated dataset saved at: {updated_file_path}")


✅ Updated dataset saved at: /Users/go82gax/Documents/Projekte/CCfD/Daten/IPCEI_combined_country_codes.csv


2. Load datasets

In [55]:
# 📥 Load both datasets
ipcei_data_path = os.path.join(WORKING_DIR, "IPCEI_combined_country_codes.csv")
orgs_data_path = os.path.join(WORKING_DIR, "organizations_linkedin.csv")

ipcei_df = pd.read_csv(ipcei_data_path)
orgs_df = pd.read_csv(orgs_data_path)

3. Prepare exact and fuzzy matching

In [57]:
from fuzzywuzzy import process, fuzz

# ✅ Keep only "name" and "Linkedin_name" from organizations dataset
orgs_df = orgs_df[['name', 'Linkedin_name']]

# ✅ Step 1: Exact Match
updated_ipcei_df = ipcei_df.merge(orgs_df, on="name", how="left")

# Identify firms that are still missing a "Linkedin_name" after exact match
unmatched_ipcei = updated_ipcei_df[updated_ipcei_df["Linkedin_name"].isna()]

# Extract organization names for fuzzy matching
org_names = orgs_df["name"].tolist()

# 🔹 Function to find the best fuzzy match
def get_best_fuzzy_match(company_name, choices, threshold=85):
    """Finds the best fuzzy match for a given company name."""
    match, score = process.extractOne(company_name, choices, scorer=fuzz.token_sort_ratio)
    return match if score >= threshold else None

# Apply fuzzy matching to find the best alternative matches
fuzzy_matches = {}

for name in unmatched_ipcei["name"]:
    best_match = get_best_fuzzy_match(name, org_names)
    if best_match:
        matched_row = orgs_df[orgs_df["name"] == best_match]
        fuzzy_matches[name] = matched_row["Linkedin_name"].values[0]

# ✅ Step 2: Apply Fuzzy Matches
updated_ipcei_df["Linkedin_name"] = updated_ipcei_df.apply(
    lambda row: fuzzy_matches.get(row["name"], row["Linkedin_name"]), axis=1
)

# 📤 Save updated dataset
updated_file_path = os.path.join(WORKING_DIR, "IPCEI_combined_with_Linkedin_fuzzy.csv")
updated_ipcei_df.to_csv(updated_file_path, index=False)

print(f"✅ Updated dataset saved at: {updated_file_path}")


✅ Updated dataset saved at: /Users/go82gax/Documents/Projekte/CCfD/Daten/IPCEI_combined_with_Linkedin_fuzzy.csv


4. Add improved matching.

In [59]:
import re

# 🔹 Function to clean and standardize company names
def clean_name(name):
    if pd.isna(name): return ""
    name = name.lower().strip()
    name = re.sub(r'\b(ag|gmbh|inc|ltd|s.a.|s.p.a.|co|group|corporation|holding|company)\b', '', name)  # Remove common words
    name = re.sub(r'[^a-z0-9 ]', '', name)  # Remove special characters
    return name.strip()

# ✅ Apply cleaning to both datasets
ipcei_df["clean_name"] = ipcei_df["name"].apply(clean_name)
orgs_df["clean_name"] = orgs_df["name"].apply(clean_name)

# ✅ Step 1: Exact Match on Cleaned Names
updated_ipcei_df = ipcei_df.merge(
    orgs_df[['clean_name', 'Linkedin_name']],
    on="clean_name",
    how="left"
)

# Identify firms still missing "Linkedin_name"
unmatched_ipcei = updated_ipcei_df[updated_ipcei_df["Linkedin_name"].isna()]

# Extract organization names for fuzzy matching
org_names = orgs_df["clean_name"].tolist()

# 🔹 Function to find best fuzzy match with multiple scorers
def get_best_fuzzy_match(company_name, choices, threshold=80):
    """Finds the best fuzzy match using multiple scoring methods."""
    match, score = process.extractOne(company_name, choices, scorer=fuzz.WRatio)  # More flexible matching
    return match if score >= threshold else None

# Apply fuzzy matching to find alternative matches
fuzzy_matches = {}

for name in unmatched_ipcei["clean_name"]:
    best_match = get_best_fuzzy_match(name, org_names)
    if best_match:
        matched_row = orgs_df[orgs_df["clean_name"] == best_match]
        fuzzy_matches[name] = matched_row["Linkedin_name"].values[0]

# ✅ Apply Fuzzy Matches where available
updated_ipcei_df["Linkedin_name"] = updated_ipcei_df.apply(
    lambda row: fuzzy_matches.get(row["clean_name"], row["Linkedin_name"]), axis=1
)

# 📤 Save updated dataset
updated_file_path = os.path.join(WORKING_DIR, "IPCEI_combined_with_Linkedin_improved.csv")
updated_ipcei_df.to_csv(updated_file_path, index=False)

print(f"✅ Updated dataset saved at: {updated_file_path}")

✅ Updated dataset saved at: /Users/go82gax/Documents/Projekte/CCfD/Daten/IPCEI_combined_with_Linkedin_improved.csv


5. Upload file after manual check and make compatibel to UTF-8 format

In [2]:
import pandas as pd

# Load the CSV with an alternative encoding (e.g., 'latin-1' or 'ISO-8859-1')
df = pd.read_csv("IPCEI_combined_with_Linkedin_improved_rechecked.csv", encoding="latin-1")

# Save the file in UTF-8 format
df.to_csv("IPCEI_clean.csv", index=False, encoding="utf-8")